<a href="https://colab.research.google.com/github/izzat-ai/learning-ai/blob/main/numpy/agricultural_monitoring.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np

In [2]:
satellite_data = np.array([
    [120, 150, 110, 450], # Hudud 1: Zich o'rmon
    [180, 130, 120, 210], # Hudud 2: Qurigan yer/Bino
    [100, 200, 90,  500], # Hudud 3: Ekin maydoni
    [255, 255, 255, 10],  # Hudud 4: Bulut (Xato ma'lumot)
    [140, 160, 130, 400]  # Hudud 5: Bog'
], dtype=float)

# Hududlarning maydoni (kvadrat metrda)
area_sq_m = np.array([5000, 2000, 7500, 1000, 3000])

satellite_data

array([[120., 150., 110., 450.],
       [180., 130., 120., 210.],
       [100., 200.,  90., 500.],
       [255., 255., 255.,  10.],
       [140., 160., 130., 400.]])

`qizil` , `yashil` , `ko'k` , `yaqin infraqizil`

In [3]:
satellite_data.shape

(5, 4)

- The NIR channel (column 4) should always be greater than the Red channel (column 1)

In [4]:
np.argmax(satellite_data[:, -1] < satellite_data[:, 0])

np.int64(3)

In [5]:
# delete the Red data that is greater than the NIR channel.
satellite_data = np.delete(satellite_data, 3, axis=0)
satellite_data

array([[120., 150., 110., 450.],
       [180., 130., 120., 210.],
       [100., 200.,  90., 500.],
       [140., 160., 130., 400.]])

In [6]:
area_sq_m = np.delete(area_sq_m, 3)
area_sq_m

array([5000, 2000, 7500, 3000])

In [7]:
# Vegetation Index Calculation
ndvi = (satellite_data[:, -1] - satellite_data[:, 0]) / (satellite_data[:, -1] + satellite_data[:, 0])
ndvi

array([0.57894737, 0.07692308, 0.66666667, 0.48148148])

In [12]:
# add the detected NDVI to the main array
np.set_printoptions(suppress=True, precision=4)
satellite_data = np.append(satellite_data, ndvi.reshape(-1, 1), axis=1)
satellite_data

array([[120.    , 150.    , 110.    , 450.    ,   0.5789],
       [180.    , 130.    , 120.    , 210.    ,   0.0769],
       [100.    , 200.    ,  90.    , 500.    ,   0.6667],
       [140.    , 160.    , 130.    , 400.    ,   0.4815]])

In [9]:
condlist = [
    ndvi > 0.5,
    (ndvi > 0.2) & (ndvi<=0.5),
    ndvi <= 0.2
]

# 'dense plant'-1.0, 'sparse plant'-0.5, 'no plant'-0.0
choicelist = [1.0, 0.5, 0.0]

new_col = np.select(condlist, choicelist)
new_col

array([1. , 0. , 1. , 0.5])

In [13]:
# append the classification answer to the main array
satellite_data = np.append(satellite_data, new_col.reshape(-1, 1), axis=1)
satellite_data

array([[120.    , 150.    , 110.    , 450.    ,   0.5789,   1.    ],
       [180.    , 130.    , 120.    , 210.    ,   0.0769,   0.    ],
       [100.    , 200.    ,  90.    , 500.    ,   0.6667,   1.    ],
       [140.    , 160.    , 130.    , 400.    ,   0.4815,   0.5   ]])

In [15]:
# green zone size
green_zone = satellite_data[:, -1]*area_sq_m
green_zone

array([5000.,    0., 7500., 1500.])